# Transformer NER trên CoNLL-2003

In [1]:
from pathlib import Path
import json
import pandas as pd
import torch
from IPython.display import display
from transformers import AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification
from ner_common import MODEL, ROOT, align_labels, load_conll, score_entities
from train_ner import run, encode_split, evaluate

data, names = load_conll()
print({split: len(data[split]) for split in data})
print('labels:', names)
if not (ROOT / 'checkpoint' / 'config.json').exists():
    run(epochs=10)
print('checkpoint:', ROOT / 'checkpoint')

{'train': 14041, 'validation': 3250, 'test': 3453}
labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
checkpoint: /home/sonnh/Documents/btvn/nlp/lab09/checkpoint


In [2]:
tokenizer = AutoTokenizer.from_pretrained(ROOT / 'checkpoint', local_files_only=True)
encoded_test = encode_split(data['test'], tokenizer)
batch = DataCollatorForTokenClassification(tokenizer)([encoded_test[0], encoded_test[1]])
model = AutoModelForTokenClassification.from_pretrained(ROOT / 'checkpoint', local_files_only=True)
model.eval()
with torch.no_grad():
    logits = model(**batch).logits
pred_ids = logits.argmax(-1)
print('input_ids', tuple(batch['input_ids'].shape), 'attention_mask', tuple(batch['attention_mask'].shape))
print('labels', tuple(batch['labels'].shape), 'logits', tuple(logits.shape))
assert logits.shape == (*batch['labels'].shape, len(names))
valid = batch['labels'][0] != -100
display(pd.DataFrame({'word': data['test'][0]['tokens'],
                      'gold': [names[i] for i in data['test'][0]['ner_tags']],
                      'prediction': [names[i] for i in pred_ids[0][valid].tolist()]}))

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/39 [00:00<?, ?it/s]

input_ids (2, 14) attention_mask (2, 14)
labels (2, 14) logits (2, 14, 9)


,word,gold,prediction
0,SOCCER,O,O
1,-,O,O
2,JAPAN,B-LOC,B-LOC
3,GET,O,O
4,LUCKY,O,O
5,WIN,O,O
6,",",O,O
7,CHINA,B-PER,B-LOC
8,IN,O,O
9,SURPRISE,O,O


In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
metrics, predictions = evaluate(model, encoded_test, data['test'], names,
                                DataCollatorForTokenClassification(tokenizer), device)
display(pd.DataFrame([metrics]))
display(pd.read_csv(ROOT / 'results.csv'))
display(pd.read_csv(ROOT / 'entity_metrics.csv'))
display(pd.read_csv(ROOT / 'entity_length_metrics.csv'))
assert len(predictions) == len(data['test'])
assert abs(metrics['f1'] - json.loads((ROOT / 'transformer_metrics.json').read_text())['f1']) < 1e-9

,precision,recall,f1,tp,fp,fn,token_precision,token_recall,token_f1,token_accuracy
0,0.77382,0.803824,0.788537,4540,1327,1108,0.83017,0.830375,0.830272,0.958566


,model,precision,recall,f1,test_sentences
0,Baseline Logistic Regression,0.694797,0.737606,0.715562,3453
1,Pretrained Transformer (google/bert_uncased_L-...,0.773820,0.803824,0.788537,3453


,entity,precision,recall,f1,tp,fp,fn
0,PER,0.851874,0.871367,0.861510,1409,245,208
1,ORG,0.702395,0.706201,0.704293,1173,497,488
2,LOC,0.844649,0.880096,0.862008,1468,270,200
3,MISC,0.608696,0.698006,0.650299,490,315,212


,length,gold,predicted,tp,precision,recall,f1
0,1,3574,3931,2929,0.745103,0.819530,0.780546
1,2,1776,1688,1450,0.859005,0.816441,0.837182
2,3,236,205,144,0.702439,0.610169,0.653061
3,4+,62,43,17,0.395349,0.274194,0.323810
